In [ ]:
# =============================================================================
# Part 1 : 데이터 로드 & 피처 엔지니어링
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import os
import gc

warnings.filterwarnings('ignore')

def set_korean_font():
    for font in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
        if font in {f.name for f in fm.fontManager.ttflist}:
            plt.rcParams['font.family'] = font
            break
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()

# ── 경로 설정 ──────────────────────────────────────────────────────────────
DATA_PATH = r'C:\Users\LG\K-Pick\data\prep\k-pick_total_v3.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {DATA_PATH}")

print("=" * 60)
print("Part 1 : 데이터 로드 & 피처 엔지니어링")
print("=" * 60)

# =============================================================================
# 1. 컬럼 확인 & 타겟 설정
# =============================================================================
print("\n1. 컬럼 확인")

cols_only = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
print(f"전체 컬럼 ({len(cols_only)}): {cols_only}")

TARGET_CANDIDATES = ['label', 'reordered', 'target', 'y']
target_col = next((c for c in TARGET_CANDIDATES if c in cols_only), None)

if target_col is None:
    raise ValueError(f"타겟 컬럼을 찾을 수 없습니다.\n현재 컬럼: {cols_only}")

print(f"타겟 컬럼: '{target_col}'")

sample_df    = pd.read_csv(DATA_PATH, nrows=1000, low_memory=False)
numeric_cols = sample_df.select_dtypes(include=[np.number]).columns.tolist()

ID_COLS   = ['user_id', 'product_id', 'order_id']
DROP_COLS = ID_COLS

use_cols = [c for c in numeric_cols if c not in DROP_COLS]
if target_col not in use_cols:
    use_cols.append(target_col)

id_cols_present = [c for c in ID_COLS if c in cols_only]
load_cols       = list(set(use_cols + id_cols_present))

print(f"로드 컬럼 ({len(load_cols)}): {load_cols}")

# =============================================================================
# 2. 데이터 로드
# =============================================================================

dtype_map = {}
for c in load_cols:
    if c == target_col:        dtype_map[c] = 'int8'
    elif c in id_cols_present: dtype_map[c] = 'int32'
    else:                      dtype_map[c] = 'float32'

# ✅ nrows 제거 → 전체 데이터 로드
df = pd.read_csv(DATA_PATH, usecols=load_cols, dtype=dtype_map, low_memory=True)
gc.collect()

print(f"shape    : {df.shape}")
print(f"메모리   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"재구매율 : {df[target_col].mean():.4f}")
print("\n클래스 분포:")
print(df[target_col].value_counts())

# =============================================================================
# 3. 피처 엔지니어링 (집계 피처)
# =============================================================================
print("\n3. 피처 엔지니어링")

if 'user_id' in df.columns:
    print("  유저 집계 피처 생성 중...")
    agg_dict = {'user_reorder_rate': (target_col, 'mean')}
    if 'order_id'   in df.columns: agg_dict['user_order_count']   = ('order_id',   'nunique')
    if 'product_id' in df.columns: agg_dict['user_product_count'] = ('product_id', 'nunique')

    user_stats = df.groupby('user_id').agg(**agg_dict).astype('float32')
    df = df.merge(user_stats, on='user_id', how='left')
    print(f"  → 유저 집계 피처 {len(agg_dict)}개 추가")

if 'product_id' in df.columns:
    print("  상품 집계 피처 생성 중...")
    product_stats = df.groupby('product_id').agg(
        product_reorder_rate = (target_col, 'mean'),
        product_order_count  = (target_col, 'count'),
    ).astype('float32')
    df = df.merge(product_stats, on='product_id', how='left')
    print(f"  → 상품 집계 피처 2개 추가")

df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()
print(f"피처 엔지니어링 후 shape: {df.shape}")

# =============================================================================
# 4. X / y 분리 & 결측치 처리
# =============================================================================
print("\n4. 전처리")

feature_cols = [c for c in df.columns if c != target_col]
X = df[feature_cols].values.astype(np.float32)
y = df[target_col].values.astype(np.int8)

del df
gc.collect()

nan_counts = np.isnan(X).sum(axis=0)
if nan_counts.sum() > 0:
    print("결측치 발견 → -1 처리:")
    for i, cnt in enumerate(nan_counts):
        if cnt > 0:
            X[np.isnan(X[:, i]), i] = -1
            print(f"  {feature_cols[i]} : {cnt}개")
else:
    print("결측치 없음")

# =============================================================================
# 5. Train / Val / Test 분할 (7:2:1)
# =============================================================================
print("\n5. 데이터 분할 (train 70% / val 20% / test 10%)")

from sklearn.model_selection import train_test_split

X_tmp,   X_test,  y_tmp,   y_test  = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)
X_train, X_val,   y_train, y_val   = train_test_split(
    X_tmp, y_tmp, test_size=float(2)/9, random_state=42, stratify=y_tmp
)

del X_tmp, y_tmp, X
gc.collect()

total = len(y_train) + len(y_val) + len(y_test)
print(f"Train : {X_train.shape[0]:,}행  ({len(y_train)/total*100:.0f}%)")
print(f"Val   : {X_val.shape[0]:,}행   ({len(y_val)/total*100:.0f}%)")
print(f"Test  : {X_test.shape[0]:,}행  ({len(y_test)/total*100:.0f}%)")
print(f"피처  : {len(feature_cols)}개")

# =============================================================================
# 6. 전체 피처 스케일링
# =============================================================================
print("\n6. 전체 피처 스케일링 (StandardScaler)")

from sklearn.preprocessing import StandardScaler

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print(f"스케일링 완료  |  Train mean≈{X_train.mean():.4f}, std≈{X_train.std():.4f}")

print("\nPart 1 완료 ✓  →  Part 2 실행 전 이 변수들을 유지하세요:")
print("  X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, scaler")

In [ ]:
# =============================================================================
# Part 2 : 모델 학습 (XGBoost · LightGBM · Logistic Regression)
# =============================================================================
 
import numpy as np
import pandas as pd
import gc
import warnings
 
warnings.filterwarnings('ignore')
 
# LR은 대용량 데이터에서 학습 속도가 느리므로 최대 30만 행으로 제한
LR_MAX_ROWS = 300_000
 
print("=" * 60)
print("Part 2 : 모델 학습")
print("=" * 60)
 
# 클래스 불균형 비율 계산: 다수 클래스 수 / 소수 클래스 수
# XGBoost의 scale_pos_weight에 사용 → 소수 클래스(재구매=1)에 가중치 부여
neg       = (y_train == 0).sum()   # 비재구매(0) 샘플 수
pos       = (y_train == 1).sum()   # 재구매(1) 샘플 수
scale_pos = neg / pos              # 불균형 비율 (예: 9.22 → 비재구매가 9.22배 많음)
print(f"\nscale_pos_weight = {scale_pos:.2f}")
 
# =============================================================================
# 공통 유틸 : 임계값 최적화 & 평가
# =============================================================================
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, classification_report
)
 
def find_best_threshold(y_true, y_prob):
    """F1 기준 최적 임계값 탐색: 0.20~0.80 사이를 0.01 단위로 순회하여 F1 최대화"""
    thresholds = np.arange(0.20, 0.80, 0.01)  # 탐색할 임계값 범위
    best_t, best_score = 0.5, 0.0             # 기본값으로 초기화
    for t in thresholds:
        pred  = (y_prob >= t).astype(int)     # 임계값 이상이면 1, 미만이면 0으로 변환
        score = f1_score(y_true, pred, zero_division=0)  # F1 계산 (분모=0일 때 0 반환)
        if score > best_score:
            best_score, best_t = score, t     # 더 높은 F1을 내는 임계값으로 갱신
    return round(best_t, 2)
 
def evaluate(name, y_true, y_pred, y_prob):
    # 기본 임계값(0.5)과 최적 임계값 모두로 예측 후 성능 비교
    best_t       = find_best_threshold(y_true, y_prob)   # F1 최적 임계값 탐색
    y_pred_tuned = (y_prob >= best_t).astype(int)        # 최적 임계값 기준 예측
 
    # 주요 분류 지표를 딕셔너리로 정리
    m = dict(
        Model          = name,
        Best_Threshold = best_t,
        AUC_ROC        = roc_auc_score(y_true, y_prob),                          # 임계값 무관한 전체 성능
        F1_default     = f1_score(y_true, y_pred,        zero_division=0),       # 기본 임계값(0.5) F1
        F1_tuned       = f1_score(y_true, y_pred_tuned,  zero_division=0),       # 최적 임계값 F1
        Precision      = precision_score(y_true, y_pred_tuned, zero_division=0), # 정밀도: TP/(TP+FP)
        Recall         = recall_score(y_true, y_pred_tuned,    zero_division=0), # 재현율: TP/(TP+FN)
        Accuracy       = accuracy_score(y_true, y_pred_tuned),                   # 전체 정확도
    )
 
    print("\n" + "=" * 60)
    print(f"[{name}]  기본 임계값 0.5  →  최적 임계값 {best_t}")
    print("=" * 60)
    print(f"  AUC-ROC       : {m['AUC_ROC']:.4f}")
    print(f"  F1 (0.5)      : {m['F1_default']:.4f}")
    print(f"  F1 ({best_t}) : {m['F1_tuned']:.4f}  ← 개선 +{m['F1_tuned']-m['F1_default']:.4f}")
    print(f"  Precision     : {m['Precision']:.4f}")
    print(f"  Recall        : {m['Recall']:.4f}")
    print("\nClassification Report (최적 임계값 기준)")
    print(classification_report(y_true, y_pred_tuned, zero_division=0))  # 클래스별 상세 리포트
    return m
 
# =============================================================================
# 2-A. XGBoost
# =============================================================================
print("\n" + "=" * 60)
print("2-A. XGBoost 학습")
print("=" * 60)
 
from xgboost import XGBClassifier
 
xgb_model = XGBClassifier(
    n_estimators          = 500,     # 최대 트리 개수 (early stopping이 조기 종료 가능)
    learning_rate         = 0.05,    # 학습률: 작을수록 안정적이나 학습 시간 증가
    max_depth             = 6,       # 트리 깊이: 깊을수록 복잡한 패턴 학습 (과적합 위험)
    min_child_weight      = 5,       # 리프 노드 최소 샘플 수: 과적합 방지
    subsample             = 0.8,     # 행 샘플링 비율: 80%만 사용 → 과적합 방지
    colsample_bytree      = 0.8,     # 컬럼 샘플링 비율: 트리마다 80% 피처만 사용
    scale_pos_weight      = scale_pos, # 클래스 불균형 보정 (소수 클래스에 가중치)
    eval_metric           = 'auc',   # 조기 종료 기준 지표: AUC
    early_stopping_rounds = 30,      # 검증셋 AUC가 30라운드 개선 없으면 학습 중단
    tree_method           = 'hist',  # 히스토그램 기반 빠른 트리 구성 (대용량에 적합)
    random_state          = 42,      # 재현성 확보를 위한 랜덤 시드
    n_jobs                = -1,      # CPU 모든 코어 사용
    verbosity             = 0,       # 학습 중 출력 메시지 숨김
)
 
# eval_set으로 검증셋 성능 모니터링하며 학습 (조기 종료 조건 체크)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"조기 종료 best round : {xgb_model.best_iteration}")  # 실제로 사용된 트리 수
 
xgb_prob    = xgb_model.predict_proba(X_test)[:, 1]  # 클래스 1(재구매)의 확률값 추출
xgb_pred    = xgb_model.predict(X_test)              # 기본 임계값(0.5) 기준 예측값
xgb_metrics = evaluate("XGBoost", y_test, xgb_pred, xgb_prob)  # 성능 평가 및 출력
 
# =============================================================================
# 2-B. LightGBM
# =============================================================================
print("\n" + "=" * 60)
print("2-B. LightGBM 학습")
print("=" * 60)
 
lgb_model   = None  # 미설치 시 None 유지 (이후 코드에서 None 체크로 건너뜀)
lgb_metrics = None
 
try:
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
 
    lgb_model = LGBMClassifier(
        n_estimators      = 500,       # 최대 트리 개수
        learning_rate     = 0.05,      # 학습률
        num_leaves        = 63,        # 트리의 최대 리프 수 (XGBoost max_depth와 다른 방식)
        min_child_samples = 20,        # 리프 노드 최소 샘플 수: 과적합 방지
        subsample         = 0.8,       # 행 샘플링 비율
        colsample_bytree  = 0.8,       # 컬럼 샘플링 비율
        is_unbalance      = True,      # 클래스 불균형 보정 (LightGBM 권장 방식, scale_pos_weight 대체)
        metric            = 'auc',     # 조기 종료 기준 지표: AUC (XGBoost와 통일)
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,        # 학습 로그 출력 끔
    )
 
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            early_stopping(30, verbose=False),  # 30라운드 개선 없으면 조기 종료
            log_evaluation(-1),                  # 로그 출력 비활성화
        ],
    )
 
    lgb_prob    = lgb_model.predict_proba(X_test)[:, 1]  # 재구매 확률
    lgb_pred    = lgb_model.predict(X_test)              # 기본 임계값 예측
    lgb_metrics = evaluate("LightGBM", y_test, lgb_pred, lgb_prob)
 
except ImportError:
    print("LightGBM 미설치 → 건너뜀  (pip install lightgbm)")  # 패키지 없으면 안전하게 건너뜀
 
# =============================================================================
# 2-C. Logistic Regression  ★수정: 별도 scaler 제거 (이미 전체 스케일링 완료)
# =============================================================================
print("\n" + "=" * 60)
print("2-C. Logistic Regression 학습")
print("=" * 60)
 
from sklearn.linear_model import SGDClassifier  # 대용량 데이터에 적합한 확률적 경사하강법 LR
 
# LR은 전체 데이터 학습이 느리므로 LR_MAX_ROWS(30만)개 샘플링
rng  = np.random.RandomState(42)
idx0 = np.where(y_train == 0)[0]  # 비재구매(0) 샘플 인덱스
idx1 = np.where(y_train == 1)[0]  # 재구매(1) 샘플 인덱스
 
# 타겟 비율에 맞춰 클래스별로 샘플 수 결정 (stratified sampling)
n1   = int(LR_MAX_ROWS * y_train.mean())  # 재구매 샘플 수 (전체의 원래 비율만큼)
n0   = LR_MAX_ROWS - n1                   # 비재구매 샘플 수
s    = np.concatenate([
    rng.choice(idx0, min(n0, len(idx0)), replace=False),  # 비재구매 랜덤 샘플링 (중복 없이)
    rng.choice(idx1, min(n1, len(idx1)), replace=False),  # 재구매 랜덤 샘플링
])
rng.shuffle(s)  # 샘플 순서 무작위 섞기 (학습 순서 편향 방지)
 
print(f"LR 샘플 수: {len(s):,}행  (class 1 비율 {y_train[s].mean():.4f})")
 
lr_model = SGDClassifier(
    loss         = 'log_loss',    # 로지스틱 손실 함수 사용 → 확률 출력 가능
    penalty      = 'l2',          # L2 정규화 (릿지): 가중치 크기 제한으로 과적합 방지
    alpha        = 1e-4,          # 정규화 강도 (클수록 강한 규제)
    max_iter     = 200,           # 최대 반복 횟수
    tol          = 1e-4,          # 수렴 허용 오차 (변화가 이 값 이하면 학습 중단)
    class_weight = 'balanced',    # 클래스 불균형 자동 보정 (소수 클래스에 높은 가중치)
    random_state = 42,
    n_jobs       = -1,
)
lr_model.fit(X_train[s], y_train[s])  # Part 1에서 이미 스케일링된 X_train 사용
 
lr_prob    = lr_model.predict_proba(X_test)[:, 1]  # 재구매 확률
lr_pred    = lr_model.predict(X_test)              # 기본 임계값 예측
lr_metrics = evaluate("Logistic Regression", y_test, lr_pred, lr_prob)
 
# =============================================================================
# 성능 비교표
# =============================================================================
print("\n" + "=" * 60)
print("전체 모델 성능 비교 (최적 임계값 기준)")
print("=" * 60)
 
# None이 아닌 모델 결과만 수집 (LightGBM 미설치 시 제외)
all_metrics = [m for m in [xgb_metrics, lgb_metrics, lr_metrics] if m is not None]
results_df  = pd.DataFrame(all_metrics).set_index('Model')
print(results_df.round(4).to_string())
 
# =============================================================================
# ★추가 : 중복 계수값 변수 확인 (가이드라인 2-c-i)
# =============================================================================
print("\n" + "=" * 60)
print("중복 계수값(importance) 변수 확인 (가이드라인 2-c-i)")
print("=" * 60)
 
# ── XGBoost 중복 importance ────────────────────────────────────────────────
xgb_imp = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': xgb_model.feature_importances_
})
xgb_dup = xgb_imp[xgb_imp.duplicated('importance', keep=False)] \
            .sort_values('importance', ascending=False)
if not xgb_dup.empty:
    print("\n[XGBoost] 중복 importance 값을 가진 변수:")
    print(xgb_dup.to_string(index=False))
else:
    print("\n[XGBoost] 중복 importance 없음")
 
# ── LightGBM 중복 importance ───────────────────────────────────────────────
if lgb_model is not None:
    lgb_imp = pd.DataFrame({
        'feature'   : feature_cols,
        'importance': lgb_model.feature_importances_
    })
    lgb_dup = lgb_imp[lgb_imp.duplicated('importance', keep=False)] \
                .sort_values('importance', ascending=False)
    if not lgb_dup.empty:
        print("\n[LightGBM] 중복 importance 값을 가진 변수:")
        print(lgb_dup.to_string(index=False))
    else:
        print("\n[LightGBM] 중복 importance 없음")
 
# ── Logistic Regression 중복 계수 ─────────────────────────────────────────
lr_coef = pd.DataFrame({
    'feature': feature_cols,
    'coef'   : lr_model.coef_[0]
})
lr_dup = lr_coef[lr_coef.duplicated('coef', keep=False)] \
           .sort_values('coef', ascending=False)
if not lr_dup.empty:
    print("\n[Logistic Regression] 중복 계수값을 가진 변수:")
    print(lr_dup.to_string(index=False))
    print("※ 중복 계수는 다중공선성(multicollinearity) 또는 동일 정보를 가진 피처 의심")
else:
    print("\n[Logistic Regression] 중복 계수 없음")
 
print("\nPart 2 완료 ✓  →  Part 3 실행 전 이 변수들을 유지하세요:")
print("  xgb_model, lgb_model, lr_model, scaler")
print("  xgb_prob, lgb_prob, lr_prob")

In [ ]:
# =============================================================================
# Part 3 : 평가 · SHAP 해석 · 시각화
# =============================================================================
 
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import (
    roc_curve, precision_recall_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score
)
 
warnings.filterwarnings('ignore')
 
def set_korean_font():
    for font in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
        if font in {f.name for f in fm.fontManager.ttflist}:
            plt.rcParams['font.family'] = font
            break
    plt.rcParams['axes.unicode_minus'] = False
 
set_korean_font()
 
print("=" * 60)
print("Part 3 : 평가 · SHAP 해석 · 시각화")
print("=" * 60)
 
# ── 모델별 확률 & 최적 임계값 정리 ──────────────────────────────────────
def find_best_threshold(y_true, y_prob):
    # Part 2와 동일한 임계값 탐색 함수 (Part 3 단독 실행 대비 재정의)
    thresholds = np.arange(0.20, 0.80, 0.01)
    best_t, best_score = 0.5, 0.0
    for t in thresholds:
        score = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return round(best_t, 2)
 
# ※ X_test는 이미 Part 1에서 스케일링 완료 → 별도 transform 불필요
# 평가할 모델과 해당 확률 예측값을 딕셔너리로 관리
models = {'XGBoost': xgb_prob, 'Logistic Regression': lr_prob}
if lgb_model is not None:
    models['LightGBM'] = lgb_prob  # LightGBM이 설치된 경우에만 추가
 
# 각 모델별 최적 임계값을 미리 계산하여 저장
best_thresholds = {name: find_best_threshold(y_test, prob) for name, prob in models.items()}
COLORS = {'XGBoost': '#E24B4A', 'LightGBM': '#1D9E75', 'Logistic Regression': '#534AB7'}  # 모델별 시각화 색상
 
# =============================================================================
# 1. ROC / PR 곡선
# =============================================================================
print("\n1. ROC / PR 곡선")
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))  # 1행 2열 서브플롯 생성
 
for name, prob in models.items():
    color = COLORS[name]
 
    # ROC 곡선: FPR(False Positive Rate) vs TPR(True Positive Rate)
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name}  AUC={auc(fpr,tpr):.4f}')
 
    # PR 곡선: Recall vs Precision (불균형 데이터에서 ROC보다 더 엄격한 평가)
    prec, rec, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name}  AUC={auc(rec,prec):.4f}')
 
axes[0].plot([0,1],[0,1],'k--',lw=0.8)  # 랜덤 분류기 기준선 (대각선)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend(loc='lower right')
 
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend(loc='upper right')
 
plt.tight_layout()
plt.show()
 
# =============================================================================
# 2. 임계값별 F1 스코어
# =============================================================================
print("\n2. 임계값 최적화 시각화")
 
# 모델 수에 맞게 서브플롯 생성
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4))
if len(models) == 1: axes = [axes]  # 모델이 1개면 리스트로 감싸서 반복문 통일
 
for ax, (name, prob) in zip(axes, models.items()):
    thresholds = np.arange(0.10, 0.90, 0.01)
    # 각 임계값에서의 F1 점수 계산 → 최적 지점 시각화
    f1_scores  = [f1_score(y_test, (prob >= t).astype(int), zero_division=0) for t in thresholds]
    best_t     = best_thresholds[name]
 
    ax.plot(thresholds, f1_scores, color=COLORS[name], lw=2)
    ax.axvline(best_t, color='black', lw=1.2, ls='--', label=f'최적 {best_t}')  # 최적 임계값 수직선
    ax.axvline(0.5,    color='gray',  lw=0.8, ls=':',  label='기본값 0.5')      # 기본 임계값 참조선
    ax.set(xlabel='Threshold', ylabel='F1 Score', title=f'{name} — Threshold vs F1', ylim=(0,1))
    ax.legend()
 
plt.tight_layout()
plt.show()
 
# =============================================================================
# 3. Confusion Matrix (최적 임계값 기준)
# =============================================================================
print("\n3. Confusion Matrix")
 
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
if len(models) == 1: axes = [axes]
 
for ax, (name, prob) in zip(axes, models.items()):
    best_t = best_thresholds[name]
    pred   = (prob >= best_t).astype(int)  # 최적 임계값 기준으로 최종 예측
    # TP/FP/FN/TN 4분면 시각화 (파란색 농도로 빈도 표현)
    disp   = ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                                    display_labels=['비재구매(0)', '재구매(1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\n(임계값 {best_t})')
 
plt.tight_layout()
plt.show()
 
# =============================================================================
# 4. 변수 중요도
# =============================================================================
print("\n4. 변수 중요도")
 
# ── 트리 모델 (XGBoost / LightGBM) ────────────────────────────────────────
tree_models = {'XGBoost': xgb_model}
if lgb_model is not None:
    tree_models['LightGBM'] = lgb_model
 
fig, axes = plt.subplots(1, len(tree_models), figsize=(12, 7))
if len(tree_models) == 1: axes = [axes]
 
for ax, (name, model) in zip(axes, tree_models.items()):
    # feature_importances_로 상위 20개 피처 추출 (수평 막대 그래프)
    imp_df = pd.DataFrame({'feature': feature_cols,
                           'importance': model.feature_importances_}) \
               .sort_values('importance', ascending=True).tail(20)  # 오름차순 후 하단 20개 = 상위 20개
 
    bars = ax.barh(imp_df['feature'], imp_df['importance'],
                   color=COLORS[name], alpha=0.8)
    ax.bar_label(bars, fmt='%.4f', fontsize=8, padding=2)  # 막대 끝에 수치 레이블 표시
    ax.set(xlabel='Importance', title=f'{name} — 상위 20 피처')
 
plt.tight_layout()
plt.show()
 
# ── Logistic Regression 계수 ───────────────────────────────────────────────
# 계수(coef) 절댓값 기준으로 영향력 크기 정렬, 양수/음수로 방향 구분
lr_coef_df = pd.DataFrame({'feature': feature_cols,
                            'coef': lr_model.coef_[0],
                            'abs_coef': np.abs(lr_model.coef_[0])}) \
               .sort_values('abs_coef', ascending=True).tail(20)
 
# 양수 계수 = 재구매 확률 증가 → 빨강, 음수 = 감소 → 초록
colors = ['#E24B4A' if c > 0 else '#1D9E75' for c in lr_coef_df['coef']]
 
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(lr_coef_df['feature'], lr_coef_df['coef'], color=colors, alpha=0.8)
ax.axvline(0, color='black', lw=0.8, ls='--')  # 0 기준선 (양수/음수 경계)
ax.set(xlabel='Coefficient',
       title='Logistic Regression — 상위 20 계수 (빨강=재구매↑, 초록=비재구매↑)')
plt.tight_layout()
plt.show()
 
# ── Logistic Regression 계수 수치 출력 ────────────────────────────────────
print("\n[Logistic Regression 전체 변수 계수]")
lr_coef_all = pd.DataFrame({'feature': feature_cols,
                             'coef': lr_model.coef_[0],
                             'abs_coef': np.abs(lr_model.coef_[0])}) \
                .sort_values('coef', ascending=False)  # 영향력이 큰 양수 계수부터 정렬
 
print(lr_coef_all[['feature', 'coef']].to_string(index=False))
print(f"\n※ 양수(+): 재구매 확률 증가  |  음수(-): 재구매 확률 감소")
 
# =============================================================================
# 5. SHAP 분석 (XGBoost, 샘플 5,000개)
# =============================================================================
print("\n5. SHAP 분석")
 
try:
    import shap  # 모델 예측의 각 피처 기여도를 설명하는 라이브러리
 
    SHAP_SAMPLE = 5_000  # 전체 테스트셋 대신 5,000개 샘플만 사용 (속도 최적화)
    idx_shap    = np.random.RandomState(42).choice(len(X_test), SHAP_SAMPLE, replace=False)
    X_shap      = X_test[idx_shap]  # SHAP 계산에 사용할 샘플 추출
 
    # TreeExplainer: 트리 기반 모델에 최적화된 SHAP 계산기 (빠르고 정확)
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_shap)  # 각 샘플·피처별 SHAP 기여도 계산
 
    # Beeswarm 플롯: 피처별 SHAP 값의 분포와 피처값(색상)을 동시에 시각화
    shap_exp = shap.Explanation(
        values        = shap_values,
        base_values   = explainer.expected_value,  # 기준값 (모델의 평균 예측값)
        data          = X_shap,
        feature_names = feature_cols,
    )
    plt.figure(figsize=(10, 7))
    shap.plots.beeswarm(shap_exp, max_display=20, show=False)  # 상위 20 피처만 표시
    plt.title('SHAP Beeswarm — XGBoost (상위 20 피처)')
    plt.tight_layout()
    plt.show()
 
    # Mean |SHAP|: 피처별 평균 절댓값 SHAP → 전체적인 영향력 크기 비교
    shap_mean_df = pd.DataFrame({'feature': feature_cols,
                                  'mean_abs_shap': np.abs(shap_values).mean(axis=0)}) \
                     .sort_values('mean_abs_shap', ascending=True).tail(20)
 
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(shap_mean_df['feature'], shap_mean_df['mean_abs_shap'],
            color='#E24B4A', alpha=0.8)
    ax.set(xlabel='Mean |SHAP value|', title='SHAP 피처 중요도 (XGBoost)')
    plt.tight_layout()
    plt.show()
 
    # XGBoost 기본 Gain 중요도와 SHAP 중요도 비교 (두 지표의 순위 차이 확인)
    cmp_df = pd.DataFrame({'feature': feature_cols,
                            'xgb_gain': xgb_model.feature_importances_,
                            'shap_mean': np.abs(shap_values).mean(axis=0)}) \
               .sort_values('shap_mean', ascending=False).head(15)
    print("\n[XGBoost Gain vs SHAP — 상위 15]")
    print(cmp_df.round(4).to_string(index=False))
 
except ImportError:
    print("SHAP 미설치 → 건너뜀  (pip install shap)")
 
# =============================================================================
# 6. 최종 성능 요약
# =============================================================================
print("\n" + "=" * 60)
print("6. 최종 성능 요약 (최적 임계값 기준)")
print("=" * 60)
 
rows = []
for name, prob in models.items():
    best_t = best_thresholds[name]
    pred   = (prob >= best_t).astype(int)  # 최적 임계값 기준 최종 예측
    rows.append({
        'Model'     : name,
        'Threshold' : best_t,
        'AUC_ROC'   : roc_auc_score(y_test, prob),
        'F1'        : f1_score(y_test, pred, zero_division=0),
        'Precision' : precision_score(y_test, pred, zero_division=0),
        'Recall'    : recall_score(y_test, pred, zero_division=0),
        'Accuracy'  : accuracy_score(y_test, pred),
    })
 
# 모든 모델 성능을 한 표로 비교 출력
print(pd.DataFrame(rows).set_index('Model').round(4).to_string())
print("\nPart 3 완료 ✓")